![](img/logo_ucm.jpg)

# MLflow Tracking

Este componente es responsable de la observabilidad. Las principales características de este módulo son el registro de métricas, artefactos y parámetros de una ejecución de MLflow. Al igual que nos ocurría con el Swagger y ReDoc que vimos en el módulo anterior, es muy útil poder navegar sobre las mlruns de manera visual.

En un entorno de producción, se utiliza como un servidor de seguimiento centralizado implementado en Python que puede ser compartido por un grupo de profesionales de ML en una organización. Esto permite que las mejoras en los modelos de ML se compartan dentro de la organización.

En el momento en el que registramos ejecuciones (*runs*) en un directorio local (*mlruns*), podemos activar la interfaz gráfica (UI) que tiene MLflow. Para ello, 

1) Desplegamos la UI, en este caso en local, sobre el puerto 5000. 

Desde terminal: 

```bash
mlflow ui --port 5000
```

Para nuestra tarea, añade la URI donde viven los experimentos, por ejemplo: 

NOTA: sustituye la ruta absoluta por la tuya.

```bash
mlflow ui --backend-store-uri /home/johndoe/Documents/software/dbx-ucm-productivizacion/notebooks_clase/tema_3_mlflow/mlruns --port 5000
```

o si lo prefieres con uv: 

```bash
uv run --package notebooks_clase mlflow ui \
  --backend-store-uri "$(pwd)/notebooks_clase/tema_3_mlflow/mlruns" \
  --host 0.0.0.0 \
  --port 5000
```

Para usuarios de Windows: 

```powershell
cd $HOME\Documents\software\dbx-ucm-productivizacion
uv run --package notebooks_clase mlflow ui `
  --backend-store-uri "$PWD\notebooks_clase\tema_3_mlflow\mlruns" `
  --host 0.0.0.0 `
  --port 5000
```


>

2) Accedemos a http://127.0.0.1:5000 


```{figure} img/mlflow/mlflow_ui.png
---
name: Experiments
width: 120%
---
MLflow UI. Experiments.
```

Esta interfaz registra todas las ejecuciones de tu experimento y permite guardar los elementos clave: parámetros, métricas, modelos y artefactos. Para cada ejecución, puedes inspeccionar y comparar las métricas y configuraciones de entrenamiento, facilitando el análisis y la selección del mejor resultado.


`````{admonition} MLflow Tracking Server
:class: important
Alternativamente, el Servidor de Seguimiento de MLflow (`Tracking Server`) permite centralizar el almacenamiento y visualización de ejecuciones, incluyendo artefactos. Puedes acceder a la interfaz web desde cualquier máquina conectada al servidor mediante la dirección `http://<IP-del-servidor>:5000`.
`````

In [2]:
!mlflow ui --port 5000

[2025-04-21 10:14:00 +0200] [139521] [INFO] Starting gunicorn 23.0.0
[2025-04-21 10:14:00 +0200] [139521] [INFO] Listening at: http://127.0.0.1:5000 (139521)
[2025-04-21 10:14:00 +0200] [139521] [INFO] Using worker: sync
[2025-04-21 10:14:00 +0200] [139522] [INFO] Booting worker with pid: 139522
[2025-04-21 10:14:00 +0200] [139523] [INFO] Booting worker with pid: 139523
[2025-04-21 10:14:00 +0200] [139524] [INFO] Booting worker with pid: 139524
[2025-04-21 10:14:00 +0200] [139525] [INFO] Booting worker with pid: 139525


Si exploramos uno de los experimentos, podemos observar todo lo que se está monitorizando en cada ejecución. Esta información es crucial para auditar y reproducir experimentos de manera eficiente, permitiendo entender y evaluar el rendimiento y los parámetros de sus modelos de ML.



```{figure} img/mlflow/mlflow_ui2.png
---
name: Details
width: 100%
---
MLflow UI. Experiments. Details.
```


Detalles: 

>
- Fechas y horas en que se inició la ejecución del experimento.
- El usuario que ejecutó el experimento.
- El status, que indica que el experimento ha finalizado, indicado por un icono verde, en este caso.
- El ID de la ejecución (*Run ID*), que es un identificador único para la ejecución.
- La duración del experimento fue de 2.3 segundos. Muy útil para modelos de Deep Learning con entrenamientos largos.
- Conjuntos de datos utilizados (*Datasets used*): Lista el conjunto de datos empleado, con un ID específico, y etiquetado como *Train*. Muy útil para reproducibilidad de experimentos.
- Las etiquetas (*Tags*) que los desarrolladores de ML pueden incluir para monitorizar información adicional.
- Los *Logged models* que indican el **flavor** del modelo registrado, que en este caso es de tipo *sklearn*.
- Los **Registered models**, que en los siguientes apartados veremos qué son. En este caso, no se muestra ningún modelo registrado en esta ejecución particular.

Además de esta información general del experimento, MLflow registra la información del algoritmo. Por tanto, por una parte, podemos explorar entre todos los argumentos del modelo, que en este caso son los de una `LogisticRegression()` de `sklearn`. Y por otra parte, el cálculo de las métricas resultantes, incluso las específicas con el conjunto de test. 

```{figure} img/mlflow/mlflow_ui3.png
---
name: Parameters & Metrics
width: 100%
---
MLflow UI. Experiments. Parameters & Metrics
```

Finalmente, accedemos a la pestaña **Artifacts**. Un artefacto se refiere a *cualquier tipo de archivo de datos* generado durante las etapas de entrenamiento, validación o prueba de un modelo de ML. Los artefactos pueden incluir, pero no están limitados a, modelos entrenados, imágenes, logs o cualquier otro archivo de salida que documente o sea necesario para entender el modelo.

`````{admonition} Artifacts
:class: important
Estos artefactos son importantes porque proporcionan una representación física del trabajo realizado durante las ejecuciones del modelo y son fundamentales para la reproducibilidad y la trazabilidad de los experimentos en MLflow. 
`````

A la derecha, observamos la ruta física donde se encuentra el artefacto e información sobre el consumo del modelo. Pero, sobre todo, en la parte izquierda, el sistema de ficheros que componen el artefacto.


```{figure} img/mlflow/mlflow_ui4.png
---
name: mlflow_ui4
align: center
---
```

Los detalles de cada artefacto los hemos acometido en detalle en el Notebook anterior.

>

`````{admonition} Autolog
:class: important
Muchos de las características exploradas en el Tracking del modelo son configurables. El hecho de haber activado el _autolog_ antes de generar el experimento nos ha permitido, por ejemplo, observar métricas que no habíamos solicitado.
`````

Además, MLflow nos propone maneras de realizar inferencia tanto con pandas como Spark a través de una UDF: 

```python

import mlflow
logged_model = 'runs:/0a09a51424c7482588cefb8efddf9557/model'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

# Predict on a Pandas DataFrame.
import pandas as pd
loaded_model.predict(pd.DataFrame(data))

```


```python

import mlflow
from pyspark.sql.functions import struct, col
logged_model = 'runs:/0a09a51424c7482588cefb8efddf9557/model'

# Load model as a Spark UDF. Override result_type if the model does not return double values.
loaded_model = mlflow.pyfunc.spark_udf(spark, model_uri=logged_model)

# Predict on a Spark DataFrame.
df.withColumn('predictions', loaded_model(struct(*map(col, df.columns))))

```